# ETF Direction Prediction with Machine Learning and Deep Learning

This Kaggle notebook is a redesigned version of the ETF daily-return project. The earlier return-regression plot was unsatisfactory because the model prediction stayed near zero while the actual returns were noisy. That is common in daily financial returns: the conditional mean is small, volatility is large, and direct next-day return regression often produces low or negative out-of-sample $R^2$.

This notebook therefore changes the project focus from **exact return magnitude prediction** to **directional prediction** and **multi-horizon prediction**.

## Literature-based design

The design combines these ideas:

1. **ETF direction prediction with deep learning**  
   Kundu and Pinsky use deep learning models such as LSTM, CNN, and RNN for daily price-direction prediction and replace individual stocks with sector ETFs and the S&P 500 because same-sector stocks are highly correlated.

2. **ETF forecasting with LSTM, GRU, CNN, and Fama-French-style factors**  
   Shih, Lin, and Day compare deep learning models against the Fama-French three-factor model for ETF daily-return prediction and find that LSTM and factor-based information are useful.

3. **Temporal Fusion Transformer idea**  
   Lim et al. introduce TFT as an attention-based architecture using recurrent layers for local temporal patterns and attention for longer-range dependencies. This notebook implements a simplified TFT-style PyTorch model, not a full production TFT.

4. **Quantile combination idea**  
   Lima and Meng propose a quantile-combination approach to reduce the effect of weak predictors and estimation errors in out-of-sample return predictability.

5. **Avoiding out-of-sample $R^2$ hacking**  
   Yae and Luo warn that out-of-sample $R^2$ can be hacked if researchers repeatedly tune models after seeing test behavior. This notebook uses fixed chronological train/validation/test splits and reports multiple metrics.

## Main changes from the previous code

- Predict **direction**: up/down instead of only exact next-day return.
- Test multiple horizons: 1-day, 5-day, and 10-day forward returns.
- Keep regression metrics only as secondary diagnostics.
- Compare classical ML and PyTorch deep learning.
- Include timers and epoch-display controls.
- Plot each ETF separately.
- Use strategy metrics: directional accuracy, ROC AUC, F1, Sharpe, max drawdown.

In [ ]:
# ============================================================
# 0. Imports and configuration
# ============================================================

import os
import time
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# -----------------------------
# Kaggle path
# -----------------------------
# Update this path to match your Kaggle dataset location.
DATA_PATH = "/kaggle/input/datasets/ngc6523/adjusted-close-prices/adjusted_close_prices.csv"

# Local fallback, useful if testing outside Kaggle.
if not os.path.exists(DATA_PATH):
    local_fallback = "/mnt/data/adjusted_close_prices.csv"
    if os.path.exists(local_fallback):
        DATA_PATH = local_fallback

# -----------------------------
# Experiment settings
# -----------------------------
TARGET_TICKERS = ["SPY", "QQQ", "GLD", "VOO"]
HORIZONS = [1, 5, 10]       # Predict 1-day, 5-day, and 10-day forward direction
LAG_DAYS = 20               # Lagged daily returns used as tabular features
LOOKBACK = 30               # Sequence length for LSTM/GRU/CNN-LSTM/TFT-style models

TEST_SIZE = 0.20
VAL_SIZE = 0.15

# Deep learning controls
RUN_DEEP_MODELS = True
SHOW_EPOCH_LOGS = False     # True = print every epoch, False = hide epoch logs
EPOCHS = 40
PATIENCE = 7
BATCH_SIZE = 64
LR = 1e-3
WEIGHT_DECAY = 1e-4

# Runtime control: set to True while debugging to run fewer experiments.
QUICK_TEST = False
if QUICK_TEST:
    TARGET_TICKERS = ["GLD"]
    HORIZONS = [1]
    EPOCHS = 5
    RUN_DEEP_MODELS = True

SEED = 42


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("DATA_PATH:", DATA_PATH)

## 1. Load adjusted close prices

Your raw data has the structure:

```text
Date | SPY | QQQ | GLD | VOO
```

The four ETFs represent:

- `SPY`: S&P 500 ETF
- `QQQ`: Nasdaq-100 ETF
- `GLD`: gold ETF
- `VOO`: Vanguard S&P 500 ETF

Note: `SPY` and `VOO` are highly similar because they both track the S&P 500. This redundancy is useful to discuss in the report.

In [ ]:
# ============================================================
# 1. Load data
# ============================================================

prices = pd.read_csv(DATA_PATH)

if "Date" not in prices.columns:
    raise ValueError("Expected a Date column in the CSV.")

prices["Date"] = pd.to_datetime(prices["Date"])
prices = prices.sort_values("Date").set_index("Date")

missing_tickers = [t for t in TARGET_TICKERS if t not in prices.columns]
if missing_tickers:
    raise ValueError(f"Missing ticker columns: {missing_tickers}. Available columns: {prices.columns.tolist()}")

prices = prices[TARGET_TICKERS].ffill().dropna()

print("Price shape:", prices.shape)
print("Date range:", prices.index.min(), "to", prices.index.max())
print("Missing values:")
print(prices.isna().sum())

print("\nHead of adjusted close prices:")
display(prices.reset_index().head())

## 2. Convert prices to stationary log returns

Daily prices are usually nonstationary. We convert adjusted close prices to log returns:

$$
r_t = \log(P_t) - \log(P_{t-1}).
$$

The plots are split by ETF, because plotting all four together makes the chart too crowded.

In [ ]:
# ============================================================
# 2. Log returns and separate plots
# ============================================================

log_prices = np.log(prices)
returns = log_prices.diff().dropna()

print("Head of daily log returns:")
display(returns.reset_index().head())

for ticker in TARGET_TICKERS:
    plt.figure(figsize=(14, 4))
    plt.plot(returns.index, returns[ticker], label=ticker, alpha=0.85)
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.title(f"Daily Log Returns - {ticker}")
    plt.xlabel("Date")
    plt.ylabel("Log Return")
    plt.legend()
    plt.grid(True)
    plt.show()

## 3. Feature engineering

For each target ETF and each horizon, we create features using only information available at time $t$.

The target forward return is:

$$
r_{t,t+h} = \log(P_{t+h}) - \log(P_t).
$$

The direction label is:

$$
y_t = \mathbf{1}\{r_{t,t+h} > 0\}.
$$

Features include:

- lagged returns of all ETFs,
- rolling mean and rolling volatility,
- momentum features,
- cross-sectional market mean return and dispersion,
- simple correlation features between the target ETF and other ETFs.

This setup avoids look-ahead bias because the features use data up to time $t$, while the target uses the future return from $t$ to $t+h$.

In [ ]:
# ============================================================
# 3. Feature engineering
# ============================================================

def make_features_for_target(prices, returns, target_ticker, horizon=1, lag_days=20):
    '''
    Create tabular features and labels for one target ETF and one forecast horizon.

    X_t uses information up to date t.
    forward_return_t = log(P_{t+h}) - log(P_t).
    direction_t = 1 if forward_return_t > 0 else 0.
    '''
    feature_df = pd.DataFrame(index=returns.index)

    # Lagged returns for all ETFs
    for ticker in returns.columns:
        for lag in range(1, lag_days + 1):
            feature_df[f"{ticker}_ret_lag_{lag}"] = returns[ticker].shift(lag)

    # Today's return for all ETFs, available after today's close
    for ticker in returns.columns:
        feature_df[f"{ticker}_ret_today"] = returns[ticker]

    # Rolling features for every ETF
    for ticker in returns.columns:
        r = returns[ticker]
        for window in [5, 10, 20, 60]:
            feature_df[f"{ticker}_roll_mean_{window}"] = r.rolling(window).mean()
            feature_df[f"{ticker}_roll_vol_{window}"] = r.rolling(window).std()
            feature_df[f"{ticker}_momentum_{window}"] = prices[ticker].pct_change(window)

    # Cross-sectional features across the 4 ETFs
    feature_df["market_mean_return"] = returns.mean(axis=1)
    feature_df["market_return_dispersion"] = returns.std(axis=1)

    # Rolling correlations of the target with other ETFs
    for ticker in returns.columns:
        if ticker != target_ticker:
            feature_df[f"corr_{target_ticker}_{ticker}_20"] = returns[target_ticker].rolling(20).corr(returns[ticker])
            feature_df[f"corr_{target_ticker}_{ticker}_60"] = returns[target_ticker].rolling(60).corr(returns[ticker])

    # Forward return and direction target
    forward_return = (log_prices[target_ticker].shift(-horizon) - log_prices[target_ticker]).rename("forward_return")
    direction = (forward_return > 0).astype(int).rename("direction")

    data = pd.concat([feature_df, forward_return, direction], axis=1).dropna()

    X = data.drop(columns=["forward_return", "direction"])
    y_direction = data["direction"]
    y_return = data["forward_return"]

    return X, y_direction, y_return, data


# Quick example
X_ex, y_dir_ex, y_ret_ex, data_ex = make_features_for_target(
    prices=prices,
    returns=returns,
    target_ticker="GLD",
    horizon=1,
    lag_days=LAG_DAYS,
)

print("Example feature matrix:", X_ex.shape)
print("Example target balance:")
print(y_dir_ex.value_counts(normalize=True).rename("proportion"))
display(data_ex.reset_index().head())

## 4. Time-based train / validation / test split

Financial time series should not be randomly split. This notebook uses a chronological split:

- earlier period: training,
- middle period: validation,
- latest period: test.

This is closer to real forecasting and reduces look-ahead bias.

In [ ]:
# ============================================================
# 4. Chronological split
# ============================================================

def time_split(X, y_direction, y_return, val_size=0.15, test_size=0.20):
    n = len(X)
    test_start = int(n * (1 - test_size))
    val_start = int(test_start * (1 - val_size))

    splits = {
        "X_train": X.iloc[:val_start],
        "y_dir_train": y_direction.iloc[:val_start],
        "y_ret_train": y_return.iloc[:val_start],

        "X_val": X.iloc[val_start:test_start],
        "y_dir_val": y_direction.iloc[val_start:test_start],
        "y_ret_val": y_return.iloc[val_start:test_start],

        "X_test": X.iloc[test_start:],
        "y_dir_test": y_direction.iloc[test_start:],
        "y_ret_test": y_return.iloc[test_start:],
    }
    return splits

## 5. Evaluation metrics

Because exact return prediction is difficult, we evaluate the models mainly by direction and trading-style diagnostics:

- Accuracy
- Precision, recall, F1
- ROC AUC
- Confusion matrix
- Long/short Sharpe ratio
- Maximum drawdown
- Regression diagnostics for return forecasts when available

The simple strategy is:

```text
If predicted probability > 0.5: long the ETF
Otherwise: short the ETF
```

This is not a realistic trading system yet because it ignores transaction costs, slippage, taxes, and shorting constraints. It is only a diagnostic.

In [ ]:
# ============================================================
# 5. Evaluation metrics
# ============================================================

def max_drawdown(equity_curve):
    equity_curve = np.asarray(equity_curve)
    running_max = np.maximum.accumulate(equity_curve)
    drawdown = equity_curve / running_max - 1.0
    return drawdown.min()


def evaluate_direction_model(y_true_dir, y_true_return, prob_up, model_name, target, horizon):
    y_true_dir = np.asarray(y_true_dir).astype(int)
    y_true_return = np.asarray(y_true_return)
    prob_up = np.asarray(prob_up)

    pred_dir = (prob_up >= 0.5).astype(int)

    accuracy = accuracy_score(y_true_dir, pred_dir)
    precision = precision_score(y_true_dir, pred_dir, zero_division=0)
    recall = recall_score(y_true_dir, pred_dir, zero_division=0)
    f1 = f1_score(y_true_dir, pred_dir, zero_division=0)

    try:
        auc = roc_auc_score(y_true_dir, prob_up)
    except Exception:
        auc = np.nan

    # Long if predicted up, short otherwise
    position = np.where(pred_dir == 1, 1, -1)
    strategy_return = position * y_true_return
    buy_hold_return = y_true_return

    if np.std(strategy_return) > 0:
        # Scale by approximate number of horizon periods per year.
        annual_factor = np.sqrt(252 / horizon)
        sharpe = np.mean(strategy_return) / np.std(strategy_return) * annual_factor
    else:
        sharpe = np.nan

    strategy_curve = np.exp(np.cumsum(strategy_return))
    buy_hold_curve = np.exp(np.cumsum(buy_hold_return))

    mdd = max_drawdown(strategy_curve)
    final_strategy_growth = strategy_curve[-1]
    final_buy_hold_growth = buy_hold_curve[-1]

    return {
        "target": target,
        "horizon": horizon,
        "model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": auc,
        "LongShort_Sharpe": sharpe,
        "Max_Drawdown": mdd,
        "Final_Strategy_Growth": final_strategy_growth,
        "Final_BuyHold_Growth": final_buy_hold_growth,
        "Mean_Forward_Return": np.mean(y_true_return),
        "Mean_Strategy_Return": np.mean(strategy_return),
    }


def evaluate_return_forecast(y_true_return, pred_return, model_name, target, horizon):
    y_true_return = np.asarray(y_true_return)
    pred_return = np.asarray(pred_return)

    mae = mean_absolute_error(y_true_return, pred_return)
    rmse = np.sqrt(mean_squared_error(y_true_return, pred_return))
    r2 = r2_score(y_true_return, pred_return)

    # Direction probability proxy from predicted return sign/magnitude.
    # This is only used to place return models in the same comparison table.
    prob_up = 1 / (1 + np.exp(-pred_return / (np.std(pred_return) + 1e-8)))
    direction_metrics = evaluate_direction_model(
        y_true_dir=(y_true_return > 0).astype(int),
        y_true_return=y_true_return,
        prob_up=prob_up,
        model_name=model_name,
        target=target,
        horizon=horizon,
    )

    direction_metrics.update({"MAE": mae, "RMSE": rmse, "R2": r2})
    return direction_metrics


def plot_confusion(y_true_dir, prob_up, title):
    pred_dir = (np.asarray(prob_up) >= 0.5).astype(int)
    cm = confusion_matrix(y_true_dir, pred_dir)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Down/Zero", "Up"])
    plt.figure(figsize=(6, 5))
    disp.plot(cmap="Blues", values_format="d")
    plt.title(title)
    plt.grid(False)
    plt.show()


def plot_strategy(y_true_return, prob_up, dates, title):
    pred_dir = (np.asarray(prob_up) >= 0.5).astype(int)
    position = np.where(pred_dir == 1, 1, -1)
    strategy_return = position * np.asarray(y_true_return)
    buy_hold_return = np.asarray(y_true_return)

    strategy_curve = np.exp(np.cumsum(strategy_return))
    buy_hold_curve = np.exp(np.cumsum(buy_hold_return))

    plt.figure(figsize=(12, 5))
    plt.plot(dates, strategy_curve, label="Model long/short")
    plt.plot(dates, buy_hold_curve, label="Buy and hold")
    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Growth of $1")
    plt.legend()
    plt.grid(True)
    plt.show()

## 6. Classical ML baselines

These models give useful baselines before deep learning:

- Logistic Regression
- Random Forest Classifier
- Gradient Boosting Classifier

We also include return-regression baselines as diagnostics:

- Ridge Regression
- Random Forest Regressor
- Gradient Boosting Regressor

The earlier flat prediction problem is expected for return regression. Direction classification is now the main task.

In [ ]:
# ============================================================
# 6. Classical models
# ============================================================

def get_classical_classifiers():
    return {
        "LogisticRegression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)),
        ]),
        "RandomForestClassifier": RandomForestClassifier(
            n_estimators=500,
            max_depth=6,
            min_samples_leaf=10,
            class_weight="balanced_subsample",
            random_state=SEED,
            n_jobs=-1,
        ),
        "GradientBoostingClassifier": GradientBoostingClassifier(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            subsample=0.8,
            random_state=SEED,
        ),
    }


def get_classical_regressors():
    return {
        "RidgeRegressor": Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0)),
        ]),
        "RandomForestRegressor": RandomForestRegressor(
            n_estimators=500,
            max_depth=6,
            min_samples_leaf=10,
            random_state=SEED,
            n_jobs=-1,
        ),
        "GradientBoostingRegressor": GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            subsample=0.8,
            random_state=SEED,
        ),
    }

## 7. PyTorch sequence datasets

The deep learning models use sequences of historical feature vectors. For example, with `LOOKBACK = 30`, the model sees the previous 30 observations and predicts the direction of the forward return.

In [ ]:
# ============================================================
# 7. Sequence dataset
# ============================================================

class DirectionSequenceDataset(Dataset):
    def __init__(self, X_df, y_direction, y_return, lookback):
        self.X = X_df.values.astype(np.float32)
        self.y_direction = y_direction.values.astype(np.float32)
        self.y_return = y_return.values.astype(np.float32)
        self.dates = X_df.index
        self.lookback = lookback

    def __len__(self):
        return len(self.X) - self.lookback + 1

    def __getitem__(self, idx):
        end = idx + self.lookback
        x_seq = self.X[idx:end]
        y_dir = self.y_direction[end - 1]
        y_ret = self.y_return[end - 1]
        date = str(self.dates[end - 1])
        return torch.tensor(x_seq), torch.tensor(y_dir), torch.tensor(y_ret), date

## 8. PyTorch deep learning models

The notebook includes:

1. **LSTM Classifier**: strong baseline for sequence data.
2. **GRU Classifier**: similar to LSTM but usually lighter.
3. **CNN-LSTM Classifier**: local temporal pattern extraction followed by sequence modeling.
4. **Simple TFT-style Classifier**: simplified model using feature projection, gated residual block, LSTM, and multi-head attention.
5. **Quantile LSTM Regressor**: predicts multiple conditional quantiles; the combined quantile average is converted into a directional signal.

In [ ]:
# ============================================================
# 8. PyTorch models
# ============================================================

class LSTMClassifier(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_layers=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.head(last).squeeze(-1)


class GRUClassifier(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_layers=1, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        out, _ = self.gru(x)
        last = out[:, -1, :]
        return self.head(last).squeeze(-1)


class CNNLSTMClassifier(nn.Module):
    def __init__(self, n_features, conv_channels=64, hidden_size=64, dropout=0.2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels=n_features, out_channels=conv_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(in_channels=conv_channels, out_channels=conv_channels, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.lstm = nn.LSTM(
            input_size=conv_channels,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        # x: batch, time, features
        z = x.transpose(1, 2)      # batch, features, time
        z = self.conv(z)           # batch, channels, time
        z = z.transpose(1, 2)      # batch, time, channels
        out, _ = self.lstm(z)
        last = out[:, -1, :]
        return self.head(last).squeeze(-1)


class GatedResidualNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout=0.2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.elu = nn.ELU()
        self.fc2 = nn.Linear(hidden_dim, input_dim)
        self.dropout = nn.Dropout(dropout)
        self.gate = nn.Linear(input_dim, input_dim)
        self.norm = nn.LayerNorm(input_dim)

    def forward(self, x):
        residual = x
        z = self.fc1(x)
        z = self.elu(z)
        z = self.dropout(z)
        z = self.fc2(z)
        g = torch.sigmoid(self.gate(residual))
        return self.norm(residual + g * z)


class SimpleTFTStyleClassifier(nn.Module):
    def __init__(self, n_features, d_model=64, n_heads=4, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.grn = GatedResidualNetwork(d_model, d_model * 2, dropout)
        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=d_model,
            num_layers=1,
            batch_first=True,
        )
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        z = self.input_proj(x)
        z = self.grn(z)
        lstm_out, _ = self.lstm(z)
        attn_out, _ = self.attn(lstm_out, lstm_out, lstm_out)
        z = self.norm(lstm_out + attn_out)
        last = z[:, -1, :]
        return self.head(last).squeeze(-1)


class QuantileLSTMRegressor(nn.Module):
    def __init__(self, n_features, quantiles=[0.1, 0.25, 0.5, 0.75, 0.9], hidden_size=64, dropout=0.2):
        super().__init__()
        self.quantiles = quantiles
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, len(quantiles)),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.head(last)


def quantile_loss(preds, target, quantiles):
    losses = []
    for i, q in enumerate(quantiles):
        errors = target - preds[:, i]
        loss = torch.max((q - 1) * errors, q * errors)
        losses.append(loss.unsqueeze(1))
    return torch.mean(torch.sum(torch.cat(losses, dim=1), dim=1))

## 9. Training utilities

Use `SHOW_EPOCH_LOGS = True` if you want to print every epoch. Otherwise, the notebook only prints model-level progress.

In [ ]:
# ============================================================
# 9. Training utilities
# ============================================================

def train_torch_classifier(model, train_loader, val_loader, verbose=False):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()

    best_val_loss = np.inf
    best_state = None
    patience_counter = 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []

        for xb, y_dir, _, _ in train_loader:
            xb = xb.to(device)
            y_dir = y_dir.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, y_dir)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, y_dir, _, _ in val_loader:
                xb = xb.to(device)
                y_dir = y_dir.to(device)
                logits = model(xb)
                loss = criterion(logits, y_dir)
                val_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if verbose:
            print(f"Epoch {epoch:03d} | train {train_loss:.8f} | val {val_loss:.8f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            if verbose:
                print("Early stopping.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


def train_torch_quantile(model, train_loader, val_loader, verbose=False):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val_loss = np.inf
    best_state = None
    patience_counter = 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []

        for xb, _, y_ret, _ in train_loader:
            xb = xb.to(device)
            y_ret = y_ret.to(device)

            optimizer.zero_grad()
            preds = model(xb)
            loss = quantile_loss(preds, y_ret, model.quantiles)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, _, y_ret, _ in val_loader:
                xb = xb.to(device)
                y_ret = y_ret.to(device)
                preds = model(xb)
                loss = quantile_loss(preds, y_ret, model.quantiles)
                val_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if verbose:
            print(f"Epoch {epoch:03d} | train {train_loss:.8f} | val {val_loss:.8f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            if verbose:
                print("Early stopping.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


def predict_torch_classifier(model, loader):
    model.eval()
    probs = []
    y_dirs = []
    y_rets = []
    dates = []

    with torch.no_grad():
        for xb, y_dir, y_ret, batch_dates in loader:
            xb = xb.to(device)
            logits = model(xb)
            prob = torch.sigmoid(logits).detach().cpu().numpy()
            probs.extend(prob)
            y_dirs.extend(y_dir.numpy())
            y_rets.extend(y_ret.numpy())
            dates.extend(batch_dates)

    return np.array(y_dirs).astype(int), np.array(y_rets), np.array(probs), pd.to_datetime(dates)


def predict_torch_quantile(model, loader):
    model.eval()
    q_preds = []
    y_dirs = []
    y_rets = []
    dates = []

    with torch.no_grad():
        for xb, y_dir, y_ret, batch_dates in loader:
            xb = xb.to(device)
            preds = model(xb).detach().cpu().numpy()
            q_preds.append(preds)
            y_dirs.extend(y_dir.numpy())
            y_rets.extend(y_ret.numpy())
            dates.extend(batch_dates)

    q_preds = np.vstack(q_preds)
    combined_return_pred = q_preds.mean(axis=1)

    # Convert combined predicted return to probability-like score.
    scale = np.std(combined_return_pred) + 1e-8
    prob_up = 1 / (1 + np.exp(-combined_return_pred / scale))

    return np.array(y_dirs).astype(int), np.array(y_rets), prob_up, combined_return_pred, q_preds, pd.to_datetime(dates)

## 10. Main experiment loop

This loop trains all selected models for each ETF and each forecast horizon.

The notebook records:

- model metrics,
- prediction arrays,
- runtime per target/horizon,
- total runtime.

In [ ]:
# ============================================================
# 10. Main experiment loop
# ============================================================

all_results = []
all_predictions = {}

experiment_start = time.time()

for target in TARGET_TICKERS:
    for horizon in HORIZONS:
        combo_start = time.time()
        key = (target, horizon)
        all_predictions[key] = {}

        print("\n" + "=" * 90)
        print(f"Target ETF: {target} | Horizon: {horizon} trading day(s)")
        print("=" * 90)

        X, y_dir, y_ret, data = make_features_for_target(
            prices=prices,
            returns=returns,
            target_ticker=target,
            horizon=horizon,
            lag_days=LAG_DAYS,
        )

        splits = time_split(X, y_dir, y_ret, val_size=VAL_SIZE, test_size=TEST_SIZE)

        X_train = splits["X_train"]
        y_dir_train = splits["y_dir_train"]
        y_ret_train = splits["y_ret_train"]

        X_val = splits["X_val"]
        y_dir_val = splits["y_dir_val"]
        y_ret_val = splits["y_ret_val"]

        X_test = splits["X_test"]
        y_dir_test = splits["y_dir_test"]
        y_ret_test = splits["y_ret_test"]

        print("Train:", X_train.index.min(), "to", X_train.index.max(), X_train.shape)
        print("Val:  ", X_val.index.min(), "to", X_val.index.max(), X_val.shape)
        print("Test: ", X_test.index.min(), "to", X_test.index.max(), X_test.shape)
        print("Train class balance:")
        print(y_dir_train.value_counts(normalize=True).sort_index())

        # ----------------------------------------------------
        # Classical classifiers
        # ----------------------------------------------------
        classifiers = get_classical_classifiers()
        for model_name, model in classifiers.items():
            print(f"Training {model_name}...")
            model.fit(X_train, y_dir_train)

            if hasattr(model, "predict_proba"):
                prob_up = model.predict_proba(X_test)[:, 1]
            else:
                scores = model.decision_function(X_test)
                prob_up = 1 / (1 + np.exp(-scores))

            metrics = evaluate_direction_model(
                y_true_dir=y_dir_test,
                y_true_return=y_ret_test,
                prob_up=prob_up,
                model_name=model_name,
                target=target,
                horizon=horizon,
            )
            all_results.append(metrics)

            all_predictions[key][model_name] = {
                "y_dir": y_dir_test.values,
                "y_ret": y_ret_test.values,
                "prob_up": prob_up,
                "dates": y_dir_test.index,
            }

        # ----------------------------------------------------
        # Classical regressors as secondary diagnostics
        # ----------------------------------------------------
        regressors = get_classical_regressors()
        for model_name, model in regressors.items():
            print(f"Training {model_name}...")
            model.fit(X_train, y_ret_train)
            pred_return = model.predict(X_test)

            metrics = evaluate_return_forecast(
                y_true_return=y_ret_test,
                pred_return=pred_return,
                model_name=model_name,
                target=target,
                horizon=horizon,
            )
            all_results.append(metrics)

            prob_up = 1 / (1 + np.exp(-pred_return / (np.std(pred_return) + 1e-8)))
            all_predictions[key][model_name] = {
                "y_dir": y_dir_test.values,
                "y_ret": y_ret_test.values,
                "prob_up": prob_up,
                "pred_return": pred_return,
                "dates": y_dir_test.index,
            }

        # ----------------------------------------------------
        # Deep learning sequence models
        # ----------------------------------------------------
        if RUN_DEEP_MODELS:
            scaler = StandardScaler()
            X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), index=X_train.index, columns=X_train.columns)
            X_val_scaled = pd.DataFrame(scaler.transform(X_val), index=X_val.index, columns=X_val.columns)
            X_test_scaled = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=X_test.columns)

            train_ds = DirectionSequenceDataset(X_train_scaled, y_dir_train, y_ret_train, LOOKBACK)
            val_ds = DirectionSequenceDataset(X_val_scaled, y_dir_val, y_ret_val, LOOKBACK)
            test_ds = DirectionSequenceDataset(X_test_scaled, y_dir_test, y_ret_test, LOOKBACK)

            train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)
            val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
            test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

            n_features = X_train.shape[1]

            torch_models = {
                "LSTMClassifier": LSTMClassifier(n_features=n_features, hidden_size=64, dropout=0.2),
                "GRUClassifier": GRUClassifier(n_features=n_features, hidden_size=64, dropout=0.2),
                "CNNLSTMClassifier": CNNLSTMClassifier(n_features=n_features, conv_channels=64, hidden_size=64, dropout=0.2),
                "SimpleTFTStyleClassifier": SimpleTFTStyleClassifier(n_features=n_features, d_model=64, n_heads=4, dropout=0.2),
            }

            for model_name, model in torch_models.items():
                print(f"Training {model_name}...")
                model, history = train_torch_classifier(
                    model,
                    train_loader,
                    val_loader,
                    verbose=SHOW_EPOCH_LOGS,
                )
                y_dir_seq, y_ret_seq, prob_up, dates_seq = predict_torch_classifier(model, test_loader)

                metrics = evaluate_direction_model(
                    y_true_dir=y_dir_seq,
                    y_true_return=y_ret_seq,
                    prob_up=prob_up,
                    model_name=model_name,
                    target=target,
                    horizon=horizon,
                )
                all_results.append(metrics)

                all_predictions[key][model_name] = {
                    "y_dir": y_dir_seq,
                    "y_ret": y_ret_seq,
                    "prob_up": prob_up,
                    "dates": dates_seq,
                    "history": history,
                }

            # Quantile LSTM return model
            print("Training QuantileLSTMRegressor...")
            q_model = QuantileLSTMRegressor(
                n_features=n_features,
                quantiles=[0.1, 0.25, 0.5, 0.75, 0.9],
                hidden_size=64,
                dropout=0.2,
            )
            q_model, q_history = train_torch_quantile(
                q_model,
                train_loader,
                val_loader,
                verbose=SHOW_EPOCH_LOGS,
            )
            y_dir_seq, y_ret_seq, prob_up, combined_return_pred, q_preds, dates_seq = predict_torch_quantile(q_model, test_loader)

            metrics = evaluate_return_forecast(
                y_true_return=y_ret_seq,
                pred_return=combined_return_pred,
                model_name="QuantileLSTMCombined",
                target=target,
                horizon=horizon,
            )
            all_results.append(metrics)

            all_predictions[key]["QuantileLSTMCombined"] = {
                "y_dir": y_dir_seq,
                "y_ret": y_ret_seq,
                "prob_up": prob_up,
                "pred_return": combined_return_pred,
                "quantile_preds": q_preds,
                "dates": dates_seq,
                "history": q_history,
            }

        combo_elapsed = time.time() - combo_start
        print(f"Finished {target}, horizon {horizon} in {combo_elapsed:.2f} seconds ({combo_elapsed / 60:.2f} minutes).")

experiment_elapsed = time.time() - experiment_start
print("\n" + "=" * 90)
print(f"Total experiment time: {experiment_elapsed:.2f} seconds ({experiment_elapsed / 60:.2f} minutes).")
print("=" * 90)

## 11. Results table

The best model is selected by directional accuracy first, then ROC AUC, then Sharpe.

Do not select a final model purely by one lucky test-period result. For a report, discuss whether the result is stable across ETFs and horizons.

In [ ]:
# ============================================================
# 11. Results table
# ============================================================

results_df = pd.DataFrame(all_results)

# Add missing regression columns for pure classifiers
for col in ["MAE", "RMSE", "R2"]:
    if col not in results_df.columns:
        results_df[col] = np.nan

sort_cols = ["target", "horizon", "Accuracy", "ROC_AUC", "LongShort_Sharpe"]
results_sorted = results_df.sort_values(sort_cols, ascending=[True, True, False, False, False])

display(results_sorted)

best_by_target_horizon = (
    results_df
    .sort_values(["target", "horizon", "Accuracy", "ROC_AUC", "LongShort_Sharpe"], ascending=[True, True, False, False, False])
    .groupby(["target", "horizon"])
    .head(1)
    .reset_index(drop=True)
)

print("Best model for each ETF and horizon:")
display(best_by_target_horizon)

results_df.to_csv("etf_direction_model_results.csv", index=False)
best_by_target_horizon.to_csv("etf_best_models_by_target_horizon.csv", index=False)
print("Saved results CSV files.")

## 12. Visualize the best model for each ETF and horizon

For each ETF and horizon, this section plots:

1. confusion matrix,
2. strategy growth curve.

In [ ]:
# ============================================================
# 12. Plots for best models
# ============================================================

for _, row in best_by_target_horizon.iterrows():
    target = row["target"]
    horizon = int(row["horizon"])
    model_name = row["model"]
    key = (target, horizon)

    pred_info = all_predictions[key][model_name]

    y_dir_true = pred_info["y_dir"]
    y_ret_true = pred_info["y_ret"]
    prob_up = pred_info["prob_up"]
    dates = pred_info["dates"]

    print("\n" + "=" * 90)
    print(f"Best for {target}, horizon {horizon}: {model_name}")
    print("=" * 90)

    plot_confusion(
        y_true_dir=y_dir_true,
        prob_up=prob_up,
        title=f"{target} Horizon {horizon}: Confusion Matrix - {model_name}",
    )

    print(classification_report(y_dir_true, (prob_up >= 0.5).astype(int), target_names=["Down/Zero", "Up"]))

    plot_strategy(
        y_true_return=y_ret_true,
        prob_up=prob_up,
        dates=dates,
        title=f"{target} Horizon {horizon}: Strategy Curve - {model_name}",
    )

## 13. Diagnostic: why return-regression predictions can look flat

If a return-regression model predicts values close to zero, that does not automatically mean the code is broken. It often means:

1. Daily forward returns are noisy.
2. The conditional mean is close to zero.
3. MSE loss rewards conservative predictions.
4. Tree ensembles average over noisy outcomes and shrink predictions toward the mean.
5. The current feature set is limited: only SPY, QQQ, GLD, and VOO are included.

For GLD, stronger covariates would likely include:

- USD index or UUP,
- VIX,
- Treasury rates or bond ETFs such as TLT/IEF,
- CPI or inflation expectations,
- SLV and USO.

For SPY/QQQ/VOO, stronger covariates would likely include:

- VIX,
- sector ETFs,
- rates,
- Fama-French factors,
- macro variables.

In [ ]:
# ============================================================
# 13. Optional: inspect return-regression predictions
# ============================================================

# Example: plot a return-regression model if it exists for GLD horizon 1.
example_key = ("GLD", 1)
example_model = "RandomForestRegressor"

if example_key in all_predictions and example_model in all_predictions[example_key]:
    pred_info = all_predictions[example_key][example_model]
    if "pred_return" in pred_info:
        plt.figure(figsize=(14, 5))
        plt.plot(pred_info["dates"], pred_info["y_ret"], label="Actual forward return", linewidth=1)
        plt.plot(pred_info["dates"], pred_info["pred_return"], label="Predicted forward return", linewidth=1)
        plt.axhline(0, linestyle="--", linewidth=1)
        plt.title("Diagnostic: GLD Return Regression Often Shrinks Toward Zero")
        plt.xlabel("Date")
        plt.ylabel("Forward log return")
        plt.legend()
        plt.grid(True)
        plt.show()

## 14. Suggested report conclusion

A good project conclusion would be:

> Predicting the exact magnitude of daily ETF returns is difficult because daily returns have low signal-to-noise ratio and the conditional mean is close to zero. This explains why regression models often produce near-flat predictions. A more appropriate formulation is directional prediction across multiple horizons. In this project, we compare classical ML models with deep learning models, including LSTM, GRU, CNN-LSTM, a simplified TFT-style model, and a quantile LSTM. The models are evaluated using directional accuracy, ROC AUC, F1 score, confusion matrices, and simple strategy diagnostics, while return-regression metrics are treated as secondary.

## References

- Kundu, T., & Pinsky, E. Predicting daily stock price directions with deep learning models. *Machine Learning with Applications*.  
- Shih, K. H., Lin, C. H., & Day, M. Y. (2024). Forecasting ETF Performance: A Comparative Study of Deep Learning Models and the Fama-French Three-Factor Model. *Mathematics*, 12(19), 3158.  
- Lim, B., Arik, S. O., Loeff, N., & Pfister, T. (2021). Temporal Fusion Transformers for Interpretable Multi-horizon Time Series Forecasting. *International Journal of Forecasting*.  
- Lima, L. R., & Meng, F. (2016). Out-of-Sample Return Predictability: A Quantile Combination Approach. *Journal of Applied Econometrics*.  
- Grova, M. (2019). Predictive Returns using Machine Learning Techniques. Netspar.  
- Yae, J., & Luo, Y. (2023). Robust monitoring machine: a machine learning solution for out-of-sample R2-hacking in return predictability monitoring. *Financial Innovation*.